In [ ]:
import pandas as pd
import json
import time
from google import genai

GEMINI_API_KEY="INPUT_GEMINI_API_KEY"

def clean_json_text(text):
    text = text.strip()
    if text.startswith("```json"):
        text = text.replace("```json", "").replace("```", "").strip()
    elif text.startswith("```"):
        text = text.replace("```", "").strip()
    return text


def enhance_sheet_description(df, sheet_name, context):

    client = genai.Client(api_key=GEMINI_API_KEY)

    rows = []

    for idx, row in df.reset_index(drop=True).iterrows():

        rows.append({
            "row_id": int(idx),
            "title": row["title"],
            "description": row["description"]
        })

    prompt = f"""
You are an educational report writer.

Enhance all rows from this sheet in one batch.

Sheet name:
{sheet_name}

Context:
{context}

Rules:
- Preserve the original meaning.
- Do not introduce new concepts.
- Use professional educational language.
- Keep each version concise.
- Return JSON only.
- Return the same row_id for each item.
- Do not skip any row.

For each row generate:

- professional
- supportive
- strength_based
- concise

Input rows:

{json.dumps(rows, ensure_ascii=False, indent=2)}

Return JSON format:

[
    {{
        "row_id": 0,
        "professional": "",
        "supportive": "",
        "strength_based": "",
        "concise": ""
    }}
]
"""

    print(f"Enhancing {sheet_name} | rows: {len(rows)}")

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    text = clean_json_text(response.text)

    result = json.loads(text)

    result_df = pd.DataFrame(result)

    enhanced_df = df.copy().reset_index(drop=True)

    enhanced_df = enhanced_df.merge(
        result_df,
        left_index=True,
        right_on="row_id",
        how="left"
    )

    enhanced_df.drop(columns=["row_id"], inplace=True)

    return enhanced_df

In [ ]:
def enhance_sheet_learning_plan(df):

    client = genai.Client(api_key=GEMINI_API_KEY)

    rows = []

    for idx, row in df.reset_index(drop=True).iterrows():

        rows.append({
            "row_id": int(idx),
            "goal": row["goal"],
            "actions": row["actions"],
            "timeline": row["timeline"],
            "success_indicator": row["success_indicator"]
        })

    prompt = f"""
You are an educational development plan writer.

Enhance all learning plans in one batch.

Rules:

- Preserve the original meaning.
- Do not introduce new actions.
- Use professional educational language.
- Keep each version concise.
- Return JSON only.
- Return the same row_id for each item.
- Do not skip any row.

For each row generate:

- professional
- supportive
- strength_based
- concise

Input rows:

{json.dumps(rows, ensure_ascii=False, indent=2)}

Return JSON format:

[
    {{
        "row_id": 0,
        "professional": "",
        "supportive": "",
        "strength_based": "",
        "concise": ""
    }}
]
"""

    print(f"Enhancing Learning Plan | rows: {len(rows)}")

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    text = clean_json_text(response.text)

    result = json.loads(text)

    result_df = pd.DataFrame(result)

    enhanced_df = df.copy().reset_index(drop=True)

    enhanced_df = enhanced_df.merge(
        result_df,
        left_index=True,
        right_on="row_id",
        how="left"
    )

    enhanced_df.drop(columns=["row_id"], inplace=True)

    return enhanced_df

In [ ]:
input_file = "library.xlsx"

sheets = pd.read_excel(input_file, sheet_name=None)

enhanced_sheets = {}

enhanced_sheets["Metric Library"] = sheets["Metric Library"]

# Interpretation
enhanced_sheets["Interpretation Library"] = enhance_sheet_description(
    sheets["Interpretation Library"],
    sheet_name="Interpretation Library",
    context="Student learning profile interpretation"
)

time.sleep(15)

# Subject
enhanced_sheets["Subject Interpretation"] = enhance_sheet_description(
    sheets["Subject Interpretation"],
    sheet_name="Subject Interpretation",
    context="Student subject performance interpretation"
)

time.sleep(15)

# Learning Plan
enhanced_sheets["Learning Plan"] = enhance_sheet_learning_plan(
    sheets["Learning Plan"]
)

time.sleep(15)

# Growth Opportunity
enhanced_sheets["Growth Opportunity"] = enhance_sheet_description(
    sheets["Growth Opportunity"],
    sheet_name="Growth Opportunity",
    context="Student growth opportunity and exceptional strengths"
)

time.sleep(15)

# Balance Consideration
enhanced_sheets["Balance Consideration"] = enhance_sheet_description(
    sheets["Balance Consideration"],
    sheet_name="Balance Consideration",
    context="Student learning balance considerations"
)

In [ ]:
output_file = "library_enhanced.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    for sheet_name, df in enhanced_sheets.items():

        df.to_excel(
            writer,
            sheet_name=sheet_name,
            index=False
        )

print(f"Saved successfully to: {output_file}")